# ⛑️ Emergency Label Fix & SMOTE Validation

Because the ASVspoof 5 protocol has 10 columns instead of 5, the original extraction script checked the wrong column (which always contained a dash `-`), resulting in every single audio file being labeled as `1` (Spoof).

This notebook **instantly fixes** your extracted CSVs by correctly re-mapping the labels using just Pandas. You **DO NOT** need to re-extract the audio!

Additionally, this notebook performs a **Class Distribution Check** and runs a **SMOTE Dry-Run Verification** to guarantee that your datasets are perfectly healthy and ready for downstream training without crashing.

In [ ]:
# 1. Install & Import Dependencies
!pip install -q imbalanced-learn pandas numpy matplotlib seaborn

from google.colab import drive
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')
print("✅ Workspace Ready.")

In [ ]:
# 2. Re-build the correct protocol map from source files
protocol_map = {}
PROTOCOL_DIR = BASE_DIR / 'protocols'

protocol_files = list(PROTOCOL_DIR.rglob('*.tsv')) + list(PROTOCOL_DIR.rglob('*.txt'))
print(f"Found {len(protocol_files)} protocol files. Parsing...")

for split_file in protocol_files:
    with open(split_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('#') or line.startswith('speaker_id') or not line.strip():
                continue
            
            line_lower = line.lower()
            if 'bonafide' in line_lower:
                label = 0
            elif 'spoof' in line_lower:
                label = 1
            else:
                continue
                
            parts = line.strip().split()
            fname = parts[1] if parts[1].endswith('.flac') else parts[1] + '.flac'
            protocol_map[fname] = label

print(f"Successfully rebuilt correct labels for {len(protocol_map):,} audio files.")

In [ ]:
bvbb

In [ ]:
# 4. SMOTE Validation Dry-Run
def dry_run_smote(csv_path):
    if not csv_path.exists():
        return
        
    print(f"\n{'='*60}\nRunning SMOTE Dry-Run Verification on {csv_path.name}...\n{'='*60}")
    df = pd.read_csv(csv_path)
    
    # Isolate features and labels
    META_COLS = {'label', 'filename', 'split'}
    feat_cols = [c for c in df.columns if c not in META_COLS]
    
    X = df[feat_cols].values.astype(np.float32)
    y = df['label'].values
    
    # Clean NaNs and Infs
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    
    classes = np.unique(y)
    print(f"Loaded features shape: {X.shape} | Unique classes: {classes}")
    
    if len(classes) < 2:
        print(f"❌ ERROR: Cannot run SMOTE! Target still only has class: {classes}")
        return
        
    print("Applying SMOTE to balance datasets to a 1:1 ratio...")
    try:
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        
        print("🎉 SMOTE DRY-RUN SUCCESSFUL!")
        print(f"  - Original shape : {X.shape} (Spoof: {(y==1).sum():,}, Bonafide: {(y==0).sum():,})")
        print(f"  - Balanced shape : {X_resampled.shape} (Spoof: {(y_resampled==1).sum():,}, Bonafide: {(y_resampled==0).sum():,})")
    except Exception as e:
        print(f"❌ SMOTE CRASHED: {str(e)}")

# Validate training CSVs
dry_run_smote(BASE_DIR / 'spectral' / 'Dataset' / 'spectral_train.csv')
dry_run_smote(BASE_DIR / 'prosodic' / 'Dataset' / 'prosodic_train.csv')